# Tesla (TSLA) Financial Analysis

Interactive walkthrough covering:
- **P&L Analysis** — margins, growth, expense breakdown
- **Budgeting** — forward budget & variance analysis
- **Forecasting** — time-series + scenario modeling
- **Sensitivity Analysis** — tornado charts & data tables

Data source: Yahoo Finance via `yfinance`

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data_fetcher import FinancialDataFetcher
from src.pl_analysis import PLAnalyzer
from src.budgeting import BudgetAnalyzer
from src.forecasting import FinancialForecaster
from src.sensitivity import SensitivityAnalyzer

pd.set_option('display.float_format', lambda x: f'${x/1e9:,.2f}B' if abs(x) > 1e6 else f'{x:,.2f}')

## 1. Data Fetching

In [2]:
fetcher = FinancialDataFetcher('TSLA')
info = fetcher.get_company_info()
income_stmt = fetcher.get_income_statement()

print(f"Company: {info['name']}")
print(f"Sector: {info['sector']} | Industry: {info['industry']}")
income_stmt[['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income']]

Company: Tesla, Inc.
Sector: Consumer Cyclical | Industry: Auto Manufacturers


,Total Revenue,Gross Profit,Operating Income,Net Income
Year,,,,
2021,NaN,NaN,NaN,NaN
2022,$81.46B,$20.85B,$13.83B,$12.58B
2023,$96.77B,$17.66B,$8.89B,$15.00B
2024,$97.69B,$17.45B,$7.76B,$7.13B
2025,$94.83B,$17.09B,$4.85B,$3.79B


## 2. P&L Analysis

In [3]:
pl = PLAnalyzer(income_stmt)
margins = pl.margin_analysis()
growth = pl.yoy_growth()

print('=== Profit Margins ===')
display(margins)
print('\n=== YoY Growth ===')
display(growth)

=== Profit Margins ===


,Gross Margin %,Operating Margin %,Net Margin %,EBITDA Margin %
Year,,,,
2021,NaN,NaN,NaN,NaN
2022,25.60,16.98,15.45,21.68
2023,18.25,9.19,15.50,15.29
2024,17.86,7.94,7.30,15.06
2025,18.03,5.11,4.00,12.41



=== YoY Growth ===


,Total Revenue YoY %,Cost Of Revenue YoY %,Gross Profit YoY %,Operating Expense YoY %,Operating Income YoY %,Net Income YoY %,EBITDA YoY %,Basic EPS YoY %,Diluted EPS YoY %
Year,,,,,,,,,
2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023,18.80,30.53,-15.31,24.90,-35.72,19.20,-16.20,17.55,19.06
2024,0.95,1.42,-1.19,10.50,-12.72,-52.46,-0.59,-52.81,-52.67
2025,-2.93,-3.12,-2.04,26.37,-37.51,-46.79,-20.02,-47.24,-47.06


## 3. Budgeting & Variance

In [4]:
budget_analyzer = BudgetAnalyzer(income_stmt)
base_year = income_stmt.index[-2]
budget = budget_analyzer.create_budget(
    base_year=base_year,
    growth_assumptions={'Total Revenue': 0.12, 'Operating Expense': 0.08},
    years_forward=2,
)

latest = income_stmt.index[-1]
if latest in budget.index:
    variance = budget_analyzer.variance_analysis(budget, latest)
    print(f'Budget vs Actual ({latest}):')
    display(variance)

Budget vs Actual (2025):


,Budget,Actual,Variance,Variance %,Status
Metric,,,,,
Total Revenue,$109.41B,$94.83B,$-14.59B,-13.33,Unfavorable
Gross Profit,$22.23B,$17.09B,$-5.14B,-23.11,Unfavorable
Operating Income,$11.77B,$4.85B,$-6.92B,-58.79,Unfavorable
Net Income,$8.83B,$3.79B,$-5.03B,-57.01,Unfavorable


## 4. Forecasting

In [5]:
forecaster = FinancialForecaster(income_stmt)
revenue_fc = forecaster.forecast_metric('Total Revenue', periods=3)
scenarios = forecaster.scenario_forecast('Total Revenue', periods=3)

print('=== Revenue Forecast ===')
display(revenue_fc)
print('\n=== Scenarios ===')
display(scenarios.pivot(index='Year', columns='Scenario', values='Total Revenue'))

=== Revenue Forecast ===


,Total Revenue,Type
Year,,
2022,$81.46B,Actual
2023,$96.77B,Actual
2024,$97.69B,Actual
2025,$94.83B,Actual
2026,$107.21B,Forecast
2027,$113.52B,Forecast
2028,$119.84B,Forecast



=== Scenarios ===


Scenario,Base,Bear,Bull
Year,,,
2026,$104.31B,$90.09B,$118.53B
2027,$114.74B,$85.58B,$148.17B
2028,$126.21B,$81.30B,$185.21B


## 5. Sensitivity Analysis

In [6]:
sensitivity = SensitivityAnalyzer(income_stmt)
tornado = sensitivity.tornado_data()
two_way = sensitivity.two_way_sensitivity()

print('=== Tornado Chart Data ===')
display(tornado)
print('\n=== Two-Way Sensitivity (Net Income, $B) ===')
display(two_way / 1e9)

=== Tornado Chart Data ===


,Driver,Low Case,High Case,Base,Range
1,Cost Of Revenue,$9.88B,$-2.29B,$3.79B,$12.16B
0,Total Revenue,$2.35B,$4.92B,$3.79B,$2.56B
2,Operating Expense,$4.75B,$2.84B,$3.79B,$1.92B



=== Two-Way Sensitivity (Net Income, $B) ===


Total Revenue,Total Revenue -15%,Total Revenue -10%,Total Revenue -5%,Total Revenue +0%,Total Revenue +5%,Total Revenue +10%,Total Revenue +15%
Operating Expense Change,,,,,,,
-15%,3.09,3.73,4.37,5.23,5.66,6.30,6.94
-10%,2.63,3.27,3.91,4.75,5.20,5.84,6.48
-5%,2.17,2.81,3.45,4.27,4.74,5.38,6.02
+0%,1.71,2.35,3.00,3.79,4.28,4.92,5.56
+5%,1.25,1.90,2.54,3.31,3.82,4.46,5.10
+10%,0.80,1.44,2.08,2.84,3.36,4.00,4.64
+15%,0.34,0.98,1.62,2.36,2.90,3.54,4.18
